# 豆包数据复算
核验日期：2026-09-20。世界银行数据文件与本 notebook 放在同一目录。早期 WTI 使用 FRED WTISPLC 的月值，现代精确核对采用 LBMA PM / EIA。世界银行 2025 年黄金来源发生变化，不能混作 LBMA PM。复算不证明宏观因果。

In [1]:
from pathlib import Path
import openpyxl, json, statistics
base = Path.cwd()
if not (base / "worldbank-annual.xlsx").exists():
    base = base / "docs/raw/2026-09-20_doubao-audit"
wb = openpyxl.load_workbook(base / "worldbank-annual.xlsx", data_only=True, read_only=True)
rows = list(wb["Annual Prices (Nominal)"].values)
h = rows[6]
gcol, ocol = h.index("Gold"), h.index("Crude oil, WTI")
prices = {r[0]: (r[gcol], r[ocol]) for r in rows[8:] if isinstance(r[0], int)}
original = {1971:9.8,1973:13.5,1980:14.7,1985:10.1,1990:11.3,1998:29.6,2000:14.8,2008:20.2,2009:25.4,2014:16.1,2020:36.8,2025:32.2}
# FRED WTISPLC monthly observations, https://fred.stlouisfed.org/data/WTISPLC
early = {1971:[3.56]*12,1973:[3.56]*7+[4.31]*5,1980:[32.5,37,38,39.5,39.5,39.5,39.5,38,36,36,36,37]}
results=[]
for y, old in original.items():
    g,o=prices[y]
    source="World Bank annual prices"
    if y in early:
        o=statistics.mean(early[y])
        source="World Bank gold / FRED WTISPLC 12-month mean"
    results.append(dict(year=y,doubao=old,gold=g,wti=o,ratio=g/o,source=source))
# LBMA PM annual averages and EIA RWTC annual observations verified 2026-09-20.
official={2020:{"gold":1769.59,"wti":39.16},2025:{"gold":3431.54,"wti":65.39}}
for v in official.values(): v["ratio"]=v["gold"]/v["wti"]
g0,p0,g1,p1=35,3.56,3431.54,65.39
formulas={
 "fixed_1971_gold_amount_at_2025_USD":p0/g0*g1,
 "2025_oil_at_1971_official_gold_USD":p1/g1*g0,
 "oil_price_in_gold_relative_to_1971":(p1/g1)/(p0/g0),
 "USD_oil_purchasing_power_relative_to_1971":p0/p1,
 "USD_gold_purchasing_power_relative_to_1971":g0/g1,
 "market_gold_41_baseline_forward":p0/41*g1,
 "market_gold_41_baseline_reverse":p1/g1*41,
 "Japan_debt_change_2008_pp":180.851-172.951,
 "Japan_debt_change_2009_pp":198.807-180.851
}
result={"retrieved":"2026-09-20","workbook_version":rows[3][0],"comparison":results,"LBMA_EIA":official,"formulas":formulas}
print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "retrieved": "2026-09-20",
  "workbook_version": "Updated on September 02, 2026",
  "comparison": [
    {
      "year": 1971,
      "doubao": 9.8,
      "gold": 41,
      "wti": 3.56,
      "ratio": 11.516853932584269,
      "source": "World Bank gold / FRED WTISPLC 12-month mean"
    },
    {
      "year": 1973,
      "doubao": 13.5,
      "gold": 97,
      "wti": 3.8725,
      "ratio": 25.048418334409295,
      "source": "World Bank gold / FRED WTISPLC 12-month mean"
    },
    {
      "year": 1980,
      "doubao": 14.7,
      "gold": 608,
      "wti": 37.375,
      "ratio": 16.267558528428093,
      "source": "World Bank gold / FRED WTISPLC 12-month mean"
    },
    {
      "year": 1985,
      "doubao": 10.1,
      "gold": 318,
      "wti": 27.8,
      "ratio": 11.43884892086331,
      "source": "World Bank annual prices"
    },
    {
      "year": 1990,
      "doubao": 11.3,
      "gold": 383,
      "wti": 24.5,
      "ratio": 15.63265306122449,
      "source": "World Bank annu